In [ ]:
# ── Starter Cell 1: Load environment variables ──────────────────────────────
from dotenv import load_dotenv   # imports the dotenv helper that reads .env files
import os                        # provides access to environment variables at runtime

load_dotenv()                    # reads .env in the current working directory and
                                 # injects each KEY=VALUE pair into os.environ

api_key = os.getenv("ANTHROPIC_API_KEY")  # retrieves the key we stored in .env;
                                           # returns None (not an error) if missing

print("API key loaded:", api_key is not None)  # True → key found; False → check .env

In [ ]:
# ── Starter Cell 2: Initialise the Anthropic client ─────────────────────────
from anthropic import Anthropic  # imports the official Python SDK client class

client = Anthropic(api_key=api_key)  # creates a reusable client instance;
                                     # all future API calls go through this object

print("Anthropic client ready")       # quick sanity-check — no network call yet

In [ ]:
# ── Starter Cell 3: Smoke test ───────────────────────────────────────────────
response = client.messages.create(      # calls the Messages endpoint
    model="claude-haiku-4-5",           # cheapest/fastest model — good for smoke tests
    # model="claude-sonnet-4-5",        # uncomment if you want a more capable model
    max_tokens=50,                      # hard ceiling on output length (in tokens)
    messages=[                          # the conversation history — one turn here
        {"role": "user", "content": "Say hello in one short sentence."}
    ]
)

print(response.content[0].text)        # content is a list of blocks; [0] is the first;
                                       # .text extracts the plain-string payload

# CCA Lab: Being Clear and Direct with the Claude API

| | |
|---|---|
| **Course** | Building with the Claude API |
| **Sub-module** | Prompt Engineering Techniques |
| **Lesson** | Being Clear and Direct |

## What this lab covers
- The five-step request lifecycle (client → server → API → model → back)
- Why API keys must never appear in client-side code
- Building clear, unambiguous prompts by specifying format, constraints, audience, and context
- The `system` parameter: purpose, architecture, and decision table
- Multi-turn conversation with messages list accumulation and token cost analysis
- Improvement cycle with programmatic pass/fail gate
- Failure mode analysis with 6 failure modes and live variance demo
- Token usage tracking including `stop_reason` and output token variance

## CCA domains exercised
- Prompt construction and clarity
- API parameter mastery
- System prompt architecture
- Evaluation and iteration
- Production reliability patterns

## Learning objectives
1. Explain every parameter in a `client.messages.create()` call
2. Use the `system` parameter correctly and describe when/why to use it
3. Build a multi-turn conversation with the messages list pattern
4. Apply a write → evaluate → improve → re-evaluate cycle with a numeric rubric
5. Identify and mitigate six common prompt failure modes
6. Track token usage including `stop_reason` across a session

## Section 1: Prerequisites and Environment Setup
## CCA objective demonstrated: Safe credential management and environment hygiene

In [ ]:
import re                          # standard-library regex module used in rubric scoring
import json                        # used to pretty-print structured data

# ── Usage log: every API call appends a record here ─────────────────────────
usage_log = []                     # list accumulates one dict per API call;
                                   # allows session-wide token analysis later

def log_usage(label, response):
    """
    Append a token-usage record to the global usage_log.

    Parameters
    ----------
    label    : str       — human-readable name for this API call
    response : Message   — the raw response object returned by client.messages.create()

    Returns
    -------
    dict  — the record that was appended (useful for inline inspection)
    """
    record = {
        "label":        label,
        "input_tokens":  response.usage.input_tokens,   # tokens consumed by the prompt
        "output_tokens": response.usage.output_tokens,  # tokens produced by the model
        "total_tokens":  response.usage.input_tokens + response.usage.output_tokens,
        "stop_reason":   response.stop_reason,          # 'end_turn' or 'max_tokens' etc.
                                                        # stop_reason tracked so we can
                                                        # detect truncation across session
    }
    usage_log.append(record)       # mutates the global list so every section can read it
    return record

print("Usage log initialised with stop_reason tracking. Current entries:", len(usage_log))

## Section 2: API Glossary and Reference
## CCA objective demonstrated: Complete parameter literacy for client.messages.create()

| Parameter | Type | Required | Description |
|---|---|---|---|
| `model` | str | ✅ | Model ID string — determines capability and cost |
| `max_tokens` | int | ✅ | Hard ceiling on output tokens — prevents runaway cost |
| `messages` | list[dict] | ✅ | Conversation history — alternating `user`/`assistant` dicts |
| `system` | str | ❌ | Standing instructions processed before the user turn |
| `temperature` | float 0-1 | ❌ | Sampling randomness — 0 = deterministic, 1 = creative |
| `stop_sequences` | list[str] | ❌ | Strings that terminate generation early |
| `stream` | bool | ❌ | Whether to stream tokens as they are generated |

### Response object fields
| Field | Meaning |
|---|---|
| `response.content` | List of content blocks (text, tool_use, etc.) |
| `response.content[0].text` | The text string from the first content block |
| `response.usage.input_tokens` | Tokens consumed by the full prompt |
| `response.usage.output_tokens` | Tokens generated in the response |
| `response.stop_reason` | Why generation stopped (`end_turn`, `max_tokens`, `stop_sequence`) |

## Section 3: Client Setup and ask() Helper
## CCA objective demonstrated: Building a reusable, annotated API wrapper with usage tracking

In [ ]:
def ask(
    user_prompt,          # the message from the human turn — required
    system=None,          # optional standing instructions; None means no system prompt
    max_tokens=512,       # default ceiling; callers can override per-call
    model="claude-haiku-4-5",  # default model; swap to sonnet for harder tasks
    label="unlabelled",   # name used in the usage_log for traceability
):
    """
    Thin wrapper around client.messages.create() that:
      1. Builds the messages list from a plain string
      2. Optionally attaches a system prompt
      3. Logs token usage with stop_reason to usage_log
      4. Returns just the response text for convenience

    Parameters
    ----------
    user_prompt : str   — the user-turn content
    system      : str   — system prompt (or None)
    max_tokens  : int   — output token ceiling
    model       : str   — Anthropic model ID
    label       : str   — tag for usage_log entry

    Returns
    -------
    str  — the model's text response
    """
    # Build keyword arguments dict so we can conditionally add 'system'
    kwargs = {
        "model":      model,       # which model to call
        "max_tokens": max_tokens,  # must always be present — API raises if absent
        "messages": [
            {"role": "user", "content": user_prompt}  # wrap plain string in message dict;
                                                      # 'role' must be 'user' or 'assistant'
        ],
    }

    if system is not None:         # only add the 'system' key when a value is given;
        kwargs["system"] = system  # passing system=None would cause an API validation error

    response = client.messages.create(**kwargs)  # ** unpacks dict into keyword args;
                                                 # equivalent to writing each kwarg explicitly

    # response.content is a LIST because Claude can return multiple content blocks
    # (e.g., text block + tool_use block). We take [0] — the first block — which is
    # always the main text response for standard text completions.
    text = response.content[0].text  # .text extracts the string from the TextBlock object

    log_usage(label, response)     # persist token data including stop_reason to usage_log

    # stop_reason == 'max_tokens' means we hit the ceiling and the response is truncated
    if response.stop_reason == "max_tokens":
        print(f"⚠️  [{label}] Response truncated — increase max_tokens if needed")

    return text                    # caller receives a plain string — easy to print/score


# Quick validation
test_reply = ask("What is 2 + 2? Answer with just the number.", label="helper_test")
print("Helper test reply:", test_reply)
print("Usage log after test:", usage_log[-1])

## Section 4: Intentional Error Demonstration
## CCA objective demonstrated: Reading API error messages and understanding required parameters

In [ ]:
# Intentionally omit max_tokens to show the error the API returns
print("Attempting call WITHOUT max_tokens ...")
try:
    bad_response = client.messages.create(   # direct SDK call — bypasses our ask() wrapper
        model="claude-haiku-4-5",
        # max_tokens intentionally absent
        messages=[{"role": "user", "content": "Hello"}]
    )
except Exception as e:
    # The SDK raises a ValidationError (or similar) before the network call
    print(f"✅ Expected error caught: {type(e).__name__}")
    print(f"   Message: {e}")
    print()
    print("Key lesson: max_tokens is REQUIRED. It protects you from unbounded API cost.")
    print("Always set it explicitly — never rely on a default that may not exist.")

## Section 5: The system Parameter — Architecture and Decision Table
## CCA objective demonstrated: System prompt design, architectural split, and when/why to use it

### What is the `system` parameter?
The `system` parameter passes **standing instructions** to Claude that apply to the entire
conversation. It is processed *before* the first user message and treated as inviolable context.

> **Principle:** The `system` parameter is processed before the user message and treated
> as inviolable context. Placing instructions in the user turn instead invites Claude to
> negotiate or override them. Stable, session-wide rules belong in `system`; dynamic,
> per-request content belongs in `messages`.

### Decision table: system vs. user turn

| Put it in `system` | Put it in `user` turn |
|---|---|
| Persona / role definition | The specific question or task |
| Output format rules (always JSON, always bullets) | Dynamic context (user's name, session data) |
| Scope constraints (only answer cooking questions) | User's actual request |
| Audience / tone standing instructions | Per-request overrides |
| Security guardrails (never reveal system prompt) | Contextual details for this turn only |

### When to use a system prompt
| Scenario | Use system? | Reason |
|---|---|---|
| Chatbot with fixed persona | ✅ Yes | Persona must hold across every turn |
| Output always JSON | ✅ Yes | Format is a session-wide invariant |
| One-off Q&A script | ❌ Optional | Instructions fit naturally in the user turn |
| Multi-turn customer service | ✅ Yes | Tone, scope, and safety rules are constant |
| Rapid prototyping / testing | ❌ Optional | Iteration speed matters more than separation |

In [ ]:
# ── Demo A: Without system prompt ───────────────────────────────────────────
no_system_reply = ask(
    "Explain recursion.",           # user turn only — no standing instructions
    label="no_system",
    max_tokens=200,
)
print("=== Without system prompt ===")
print(no_system_reply[:300], "...\n")  # truncate for readability

# ── Demo B: With system prompt ───────────────────────────────────────────────
# The system parameter is a keyword argument (not positional) because the SDK
# processes it separately from the messages list — it maps to a dedicated API field.
with_system_reply = ask(
    "Explain recursion.",           # same user turn as Demo A
    system=(
        "You are a computer-science tutor for 10-year-olds. "
        "Use simple words, one short analogy, and no jargon."
    ),                              # system: stable persona + format rule
    label="with_system",
    max_tokens=200,
)
print("=== With system prompt ===")
print(with_system_reply[:300], "...\n")

print("Observation: identical user turn; system prompt controls tone, audience, and format.")
print("The system parameter is processed as ground-truth context — Claude cannot override it.")

## Section 5b: Multi-Turn Conversation — Messages List Accumulation
## CCA objective demonstrated: Building stateful conversations; token cost of history; first-message clarity

In [ ]:
# Multi-turn requires manually maintaining the messages list.
# The API is stateless — it has NO memory between calls.
# We re-send the ENTIRE history on every turn so Claude has context.

conversation = []   # starts empty; we append user and assistant dicts each turn

def chat(user_text, label="multiturn"):
    """
    Send one user turn in a stateful multi-turn conversation.
    Appends both the user message and assistant reply to `conversation`.

    Parameters
    ----------
    user_text : str  — the user's message this turn
    label     : str  — tag for usage_log

    Returns
    -------
    str  — the assistant's reply text
    """
    conversation.append({           # add user message to history
        "role": "user",
        "content": user_text        # first turn MUST be fully self-contained:
                                    # Claude has no memory outside this list
    })

    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=300,
        messages=conversation,      # send ENTIRE history — the API is stateless
    )

    reply = response.content[0].text   # extract text from first content block
    conversation.append({              # store assistant reply so next turn has context
        "role": "assistant",
        "content": reply
    })

    log_usage(label, response)         # track tokens including stop_reason
    return reply


# ── Three-turn conversation ───────────────────────────────────────────────────
# Turn 1: self-contained first message (no prior context exists)
r1 = chat("I'm learning Python. What is a list?", label="multiturn_t1")
print("Turn 1:", r1[:200], "...\n")

# Turn 2: refers back to Turn 1 — only works because history is re-sent
r2 = chat("Can you show me a simple example?", label="multiturn_t2")
print("Turn 2:", r2[:200], "...\n")

# Turn 3: builds further on the accumulated context
r3 = chat("How is a list different from a tuple?", label="multiturn_t3")
print("Turn 3:", r3[:200], "...\n")

print(f"Messages in conversation list after 3 turns: {len(conversation)}")
print("(6 dicts = 3 user + 3 assistant)\n")

# ── Token cost of history accumulation analysis ───────────────────────────────
mt_records = [r for r in usage_log if 'multiturn' in r['label']]
# Filter usage_log to only multi-turn records using a list comprehension:
# - iterates over every record dict in usage_log
# - keeps only those whose 'label' field contains the substring 'multiturn'

print("Token cost of accumulation:")
for i, rec in enumerate(mt_records):
    # Calculate delta: how many MORE input tokens this turn sent vs. the previous turn
    delta = rec['input_tokens'] - (mt_records[i-1]['input_tokens'] if i > 0 else 0)
    # i > 0 guard prevents index -1 wrapping on the first turn
    print(f"  Turn {i+1}: {rec['input_tokens']} input tokens (+{delta} from prior turn)")

print()
print("Insight: each turn re-sends ALL prior messages. Input tokens grow linearly.")
print("Architectural insight: the first user message must be fully self-contained —")
print("Claude has no memory outside the messages list you provide.")
print("Production implication: long conversations become expensive; consider summarisation.")

## Section 6: Core Concept — Clarity Dimensions
## CCA objective demonstrated: Operationalising prompt clarity into measurable, scorable dimensions

In [ ]:
# The four clarity dimensions we will score in the rubric
CLARITY_DIMENSIONS = [
    "format",       # Does the prompt specify output format (bullets, JSON, prose)?
    "constraints",  # Does it set limits (word count, sentence count, scope)?
    "audience",     # Does it name the target reader (expert, child, manager)?
    "context",      # Does it provide background needed to answer well?
]

# Vague baseline prompt — deliberately missing all four dimensions
vague_prompt = "Tell me about sorting algorithms."

# Clear prompt — specifies all four dimensions explicitly
clear_prompt = (
    "Explain three sorting algorithms (bubble, merge, quicksort) "
    "in exactly 3 bullet points each. "
    "Audience: a software engineering intern with 1 month of Python experience. "
    "For each algorithm include: time complexity, a one-sentence intuition, "
    "and a real-world use-case."
)

print("Vague prompt:")
print(" ", vague_prompt)
print()
print("Clear prompt:")
print(" ", clear_prompt)
print()
print("Clarity dimensions present in clear prompt:")
for d in CLARITY_DIMENSIONS:
    print(f"  ✅ {d}")

## Section 7: Improvement Cycle — Write → Evaluate → Improve → Re-evaluate
## CCA objective demonstrated: Iterative prompt engineering with numeric rubric and programmatic pass/fail gate

> **Grader reliability note:** Deterministic rubrics can over-score responses that
> contain the right words but wrong semantics. For production evals, complement
> keyword checks with a Claude-as-judge semantic pass — the rule + judge layering
> pattern from the evaluation labs.

In [ ]:
# ── Objective rubric ──────────────────────────────────────────────────────────
# Each criterion is a dict with: name, max_score, and a scorer function.
# scorer(text) → int   returns 0..max_score for that criterion.

def score_bullet_points(text):
    """
    Score 0-5: counts lines that start with a bullet marker.
    re.findall returns a list of all matches; len() converts to count.
    We use re.MULTILINE so ^ matches the start of each line, not just the string.
    """
    bullets = re.findall(r'^[\-\*•]\s', text, re.MULTILINE)
    # re.findall: we use search-style findall (not re.match) because matches can appear
    # anywhere in the multi-line string — re.match only checks the very first character.
    count = len(bullets)           # number of bullet-starting lines found
    return min(count, 5)           # cap at 5 so score stays within 0-5 range

def score_algorithms_mentioned(text):
    """
    Score 0-6: 2 points per algorithm named (bubble, merge, quicksort).
    \b word boundaries prevent 'mergesort' matching 'merge' mid-word.
    re.search (not re.match) because the word can appear anywhere in text.
    int(bool(re.search(...))) converts Match|None → True|False → 1|0.
    """
    algorithms = ["bubble", "merge", "quicksort"]
    score = 0
    for algo in algorithms:
        found = int(bool(re.search(rf'\b{algo}\b', text, re.IGNORECASE)))
        # int(bool(...)): bool() turns Match object → True, None → False;
        # int() converts True→1, False→0 so we can sum cleanly
        score += found * 2         # 2 points per algorithm found
    return score                   # max = 6 (3 algorithms × 2 points)

def score_complexity_mentioned(text):
    """
    Score 0-3: checks for Big-O notation (e.g., O(n), O(n log n), O(n²)).
    The pattern r'O\([^)]+\)' matches 'O(' followed by any chars until ')'.
    [^)]+ means 'one or more characters that are NOT a closing paren'.
    re.findall returns all matches; capped at 3 (one per algorithm).
    """
    matches = re.findall(r'O\([^)]+\)', text)  # find all Big-O expressions
    return min(len(matches), 3)    # cap at 3

def score_length_appropriate(text):
    """
    Score 0-3: response should be substantive (>100 words) but not padded (<=600 words).
    text.split() tokenises on whitespace; len() counts resulting words.
    """
    word_count = len(text.split())  # .split() with no args splits on any whitespace
    if 100 <= word_count <= 600:
        return 3
    elif word_count > 600:
        return 1                    # penalise padding
    else:
        return 0                    # too short to be useful

RUBRIC = [
    {"name": "Bullet points present",       "scorer": score_bullet_points,        "max": 5},
    {"name": "All 3 algorithms named",       "scorer": score_algorithms_mentioned, "max": 6},
    {"name": "Complexity notation present",  "scorer": score_complexity_mentioned, "max": 3},
    {"name": "Length appropriate",           "scorer": score_length_appropriate,   "max": 3},
]  # total possible = 5+6+3+3 = 17; we normalise to /20 for display

PASS_THRESHOLD = 15                # minimum total score (out of 17) to pass
print(f"Rubric defined. PASS_THRESHOLD = {PASS_THRESHOLD}/17")
print("Criteria:", [r['name'] for r in RUBRIC])

In [ ]:
def evaluate(text, label=""):
    """
    Run all rubric scorers against `text` and return a results dict.

    Returns dict with keys: criterion names + 'total'
    """
    scores = {}
    for criterion in RUBRIC:
        scores[criterion['name']] = criterion['scorer'](text)  # call each scorer function
    scores['total'] = sum(scores.values())                     # sum all criterion scores
    if label:
        print(f"\n--- Scores for: {label} ---")
        for name, score in scores.items():
            if name == 'total':
                continue
            max_s = next(r['max'] for r in RUBRIC if r['name'] == name)  # look up max
            print(f"  {name}: {score}/{max_s}")
        print(f"  TOTAL: {scores['total']}/17")
    return scores

# ── STEP 1: Write — generate vague_v0 ────────────────────────────────────────
print("=" * 60)
print("STEP 1: Write (vague prompt)")
vague_response = ask(vague_prompt, label="vague_v0", max_tokens=600)
print(vague_response[:400], "...\n")

# ── STEP 2: Evaluate vague_v0 ────────────────────────────────────────────────
print("STEP 2: Evaluate")
vague_scores = evaluate(vague_response, label="vague_v0")

# ── STEP 3: Improve — generate improved_v1 ───────────────────────────────────
print("\nSTEP 3: Improve (clear prompt)")
improved_response = ask(clear_prompt, label="improved_v1", max_tokens=600)
print(improved_response[:400], "...\n")

# ── STEP 4: Re-evaluate improved_v1 ──────────────────────────────────────────
print("STEP 4: Re-evaluate")
improved_scores = evaluate(improved_response, label="improved_v1")

# ── Comparison summary table ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"{'Criterion':<35} {'Vague':>6} {'Improved':>9}")
print("-" * 55)
for criterion in RUBRIC:
    name = criterion['name']
    print(f"{name:<35} {vague_scores[name]:>6} {improved_scores[name]:>9}")
print("-" * 55)
print(f"{'TOTAL':<35} {vague_scores['total']:>6} {improved_scores['total']:>9}")

# ── Programmatic pass/fail gate ───────────────────────────────────────────────
# This is the explicit conditional logic gate that determines whether to iterate.
# PASS_THRESHOLD was defined in the rubric cell above.
print()
if improved_scores['total'] >= PASS_THRESHOLD:
    print(f"✅ PASS — score {improved_scores['total']}/17 meets threshold {PASS_THRESHOLD}")
else:
    print(f"❌ FAIL — score {improved_scores['total']}/17 below threshold {PASS_THRESHOLD}. Iterate.")
    print("   Action: inspect lowest-scoring criteria and refine the prompt further.")

## Section 8: Failure Mode Analysis
## CCA objective demonstrated: Identifying, categorising, and mitigating six prompt failure modes

| Failure Mode | Cause | Symptom | Fix |
|---|---|---|---|
| **Format drift** | No format specified | Response structure varies per run | Specify format in system prompt |
| **Audience mismatch** | No audience named | Wrong vocabulary level for reader | Name audience explicitly |
| **Verbosity blowout** | No length constraint | Padded, over-long responses | Add word/sentence count constraint |
| **Context collapse** | Missing background | Model makes wrong assumptions | Provide relevant context upfront |
| **Truncation** | max_tokens too low | Response cut mid-sentence | Increase max_tokens; check stop_reason |
| **Instruction conflict** | Two contradictory instructions in same prompt | Claude picks one arbitrarily; output varies per run | Move invariants to system prompt; keep user turn to one task |

In [ ]:
# ── Live failure mode demo: vague prompt run 3× to measure variance ───────────
# A vague prompt run multiple times produces different word counts and formats.
# This is a PRODUCTION BUG — non-deterministic output, not an aesthetic preference.

open_prompt = "Tell me about machine learning."  # intentionally vague — no constraints

vague_runs = []                     # collect 3 responses to the same vague prompt
for i in range(3):
    r = ask(open_prompt, label=f"failure_open_run{i+1}", max_tokens=600)
    vague_runs.append(r)
    # Each call may produce different length, structure, and focus areas

# Measure word count variance across runs
word_counts = [len(r.split()) for r in vague_runs]
# List comprehension: for each response string r, split into words and count;
# produces a list of 3 integers representing word counts for runs 1, 2, 3

# Measure format variance: does each run use numbered lists?
formats = [
    "numbered" if re.search(r'\b1[.)]\s', r) else "prose"
    # re.search (not re.match): the '1.' or '1)' marker can appear ANYWHERE in the text,
    # not just at the very start. re.match only checks the beginning of the string.
    # \b word boundary ensures we match '1.' as a list marker, not '21.'
    for r in vague_runs
]

print("=" * 60)
print("FAILURE MODE: Vague prompt run 3× (production variance demo)")
print(f"Word counts across 3 runs: {word_counts}  range={max(word_counts)-min(word_counts)}")
print(f"Format per run: {formats}")
print()
print("⚠️  Production bug: the same prompt produces different word counts and formats.")
print("This is a reliability defect, not an aesthetic preference.")
print("Root cause: no format or length constraints in the prompt.")
print("Fix: add explicit constraints to the system prompt (format, length, scope).")

# ── Constrained prompt for comparison ────────────────────────────────────────
constrained_prompt = (
    "Define machine learning in exactly 3 bullet points. "
    "Each bullet: max 20 words. Audience: a non-technical manager."
)
constrained_reply = ask(constrained_prompt, label="failure_constrained", max_tokens=300)
print()
print("--- Constrained prompt reply ---")
print(constrained_reply)
print(f"\nConstrained word count: {len(constrained_reply.split())}")
print("Constrained prompts produce predictable, reproducible output — a production requirement.")

## Section 9: Token Usage Analysis
## CCA objective demonstrated: Session-wide token accounting, stop_reason tracking, and cost variance analysis

In [ ]:
# ── Session-wide token report ────────────────────────────────────────────────
print("=" * 65)
print(f"{'Label':<30} {'In':>6} {'Out':>6} {'Total':>7} {'Stop':>10}")
print("-" * 65)

session_input = 0
session_output = 0

for rec in usage_log:
    print(
        f"{rec['label']:<30} "
        f"{rec['input_tokens']:>6} "
        f"{rec['output_tokens']:>6} "
        f"{rec['total_tokens']:>7} "
        f"{rec['stop_reason']:>10}"
    )
    session_input  += rec['input_tokens']
    session_output += rec['output_tokens']

print("-" * 65)
print(f"{'SESSION TOTAL':<30} {session_input:>6} {session_output:>6} {session_input+session_output:>7}")

# ── Output token variance: vague vs. improved ────────────────────────────────
# Demonstrates that tighter prompts produce fewer output tokens, reducing cost.
vague_out    = next((r['output_tokens'] for r in usage_log if r['label'] == 'vague_v0'),    None)
improved_out = next((r['output_tokens'] for r in usage_log if r['label'] == 'improved_v1'), None)
# next(..., None): returns the first matching record's output_tokens, or None if not found.
# Generator expression inside next() is lazy — stops at first match, no full-list scan.

if vague_out and improved_out:
    delta = improved_out - vague_out
    print(f"\nOutput token variance (vague_v0 vs improved_v1): {vague_out} → {improved_out} ({delta:+d})")
    if delta < 0:
        print("Interpretation: a tighter prompt produces fewer output tokens, reducing cost.")
    elif delta > 0:
        print("Interpretation: improved prompt elicited more structured content (expected for detailed rubric).")
    else:
        print("Interpretation: output token count unchanged between prompt versions.")

# ── Stop reason summary ───────────────────────────────────────────────────────
print("\nStop reasons across session:")
for rec in usage_log:
    icon = "✅" if rec['stop_reason'] == "end_turn" else "⚠️ "
    # end_turn  → model finished naturally (good)
    # max_tokens → model was cut off (may indicate truncation)
    # stop_sequence → a stop string was hit (intentional or surprising)
    print(f"  {icon} {rec['label']}: {rec['stop_reason']}")

print("\nKey insight: all 'end_turn' stop reasons indicate complete, un-truncated responses.")
print("'max_tokens' stop reason is a signal to increase max_tokens or shorten the prompt.")

## Section 10: Practice Drills
## CCA objective demonstrated: Independent application of clear-prompt principles

In [ ]:
# ── Drill 1: Add all four clarity dimensions to a vague prompt ───────────────
print("DRILL 1: Improve this vague prompt using all four clarity dimensions")
drill1_vague = "Explain cloud computing."

# TODO: Fill in YOUR improved version with format, constraints, audience, context
drill1_improved = (
    "Explain cloud computing in exactly 4 bullet points. "
    "Each bullet: max 25 words. "
    "Audience: a small business owner with no IT background. "
    "Context: they are deciding whether to move from local servers to the cloud."
)

d1_reply = ask(drill1_improved, label="drill1", max_tokens=300)
d1_scores = evaluate(d1_reply, label="drill1")

print("\nDrill 1 reply:")
print(d1_reply)

# ── Drill 2: Use the system parameter to enforce JSON output ─────────────────
print("\n" + "=" * 60)
print("DRILL 2: Enforce JSON output via the system parameter")

d2_reply = ask(
    "List the three primary colours.",
    system="You are a data API. Always respond with valid JSON only. No prose, no markdown fences.",
    label="drill2",
    max_tokens=100,
)
print("Reply:", d2_reply)

# Verify it's valid JSON
try:
    parsed = json.loads(d2_reply)   # json.loads raises ValueError if not valid JSON
    print("✅ Valid JSON — system prompt successfully enforced format")
except (json.JSONDecodeError, ValueError):
    print("⚠️  Not valid JSON — refine system prompt to be more explicit")

# ── Drill 3: Detect a truncated response ────────────────────────────────────
print("\n" + "=" * 60)
print("DRILL 3: Intentionally trigger max_tokens truncation")

d3_reply = ask(
    "Write a detailed 500-word essay about the history of the internet.",
    label="drill3_truncate",
    max_tokens=50,   # intentionally too small to complete the essay
)
d3_record = next(r for r in usage_log if r['label'] == 'drill3_truncate')
print(f"stop_reason: {d3_record['stop_reason']}")
print(f"output_tokens: {d3_record['output_tokens']}")
print("Lesson: always check stop_reason in production to detect truncated responses.")

## Section 11: CCA Takeaways
## CCA objective demonstrated: Exam-ready mental models for clear and direct prompting

| # | Mental Model | Exam-Ready Formulation |
|---|---|---|
| 1 | **Clarity dimensions** | Every production prompt should specify Format, Constraints, Audience, and Context (FCAC). Missing any dimension degrades reliability. |
| 2 | **system vs. user split** | Stable, session-wide rules (persona, format, guardrails) belong in `system`. Dynamic, per-request content belongs in `messages`. Mixing them degrades reliability. |
| 3 | **Messages list is stateless** | The API has no memory. Every turn must re-send the full conversation history. Input tokens grow linearly; plan for summarisation in long sessions. The first message must always be fully self-contained. |
| 4 | **stop_reason as a reliability signal** | `end_turn` = complete response. `max_tokens` = truncation bug. Always log and check `stop_reason` in production systems. |
| 5 | **Deterministic rubrics + semantic judges** | Keyword rubrics are fast but can over-score wrong-semantic responses. Layer in a Claude-as-judge semantic pass for production evaluation reliability. |

---

### Architecture checklist before shipping a prompt to production
- [ ] All four clarity dimensions present (format, constraints, audience, context)
- [ ] Stable instructions in `system`; dynamic content in `messages`
- [ ] `max_tokens` set explicitly with headroom above expected output length
- [ ] `stop_reason` logged and monitored in production
- [ ] Rubric score ≥ PASS_THRESHOLD before promoting to production
- [ ] Same prompt tested ≥ 3 runs to verify output consistency (no variance bug)